# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Accessing key details without subscripting the metadata object
print(f"Dataset Name: {dataset.metadata.name}")
print(f"Dataset Description: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Version: {dataset.metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# Get record sets from the dataset metadata
record_sets = dataset.metadata.recordSet
print(f"Number of record sets: {len(record_sets)}")

# List all record set @id and their available fields:
all_recordset_ids = []
for rs in record_sets:
    print(f"\nRecord Set @id: {rs['@id']}")
    all_recordset_ids.append(rs['@id'])
    # List fields in this record set
    if 'field' in rs:
        print("Fields:")
        for field in rs['field']:
            if isinstance(field, dict) and '@id' in field:
                print(f"  - {field['@id']}")
            elif isinstance(field, str):
                print(f"  - {field}")
    else:
        print("  (No fields found)")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# We'll use the first record set as an example (replace with correct @id if known)
if all_recordset_ids:
    record_sets_to_load = all_recordset_ids
else:
    # Example fallback if no record sets found
    record_sets_to_load = []

dataframes = {}
for record_set_id in record_sets_to_load:
    print(f"Loading records for record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    # Use DataFrame only if records list is not empty
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f" - Columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f" - No records found for {record_set_id}")

# Pick the first loaded dataframe for analysis
main_record_set_id = record_sets_to_load[0] if len(record_sets_to_load) > 0 else None
if main_record_set_id:
    print(f"Available columns in `{main_record_set_id}` DataFrame:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes typical EDA tasks like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
import warnings

# Example: pick a numeric field for demonstration

if main_record_set_id:
    df = dataframes[main_record_set_id].copy()
    # Try to identify numeric columns
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_cols:
        # Try alternative detection if types are object but fields may be numeric
        for c in df.columns:
            try:
                df[c] = pd.to_numeric(df[c])
            except (ValueError, TypeError):
                continue
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()

    if numeric_cols:
        numeric_field = numeric_cols[0]
        print(f"Selecting numeric field for analysis: {numeric_field}")
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", category=RuntimeWarning)
            filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std()!=0 else 1)

        print(f"Normalized {numeric_field} (z-score) for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Try grouping by a categorical field
        group_field = None
        non_numeric_cols = [c for c in df.columns if c not in numeric_cols]
        if len(non_numeric_cols) > 0:
            group_field = non_numeric_cols[0]  # Pick the first non-numeric as example
        if group_field and group_field in df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field} (showing mean values):")
            display(grouped_df.head())
    else:
        print("No numeric columns found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

if main_record_set_id and numeric_cols:
    # Histogram for the first numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(dataframes[main_record_set_id][numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    # Boxplot by group field if available
    if group_field and group_field in dataframes[main_record_set_id].columns:
        plt.figure(figsize=(10, 4))
        sns.boxplot(x=group_field, y=numeric_field, data=dataframes[main_record_set_id].dropna(subset=[group_field, numeric_field]))
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=30, ha='right')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

_This notebook demonstrated how to load and explore a multi-record set dataset described by a Croissant schema, referencing all entities using their `@id` as per best practices. Using `mlcroissant`, we reviewed available record sets, loaded data into pandas DataFrames, performed numeric transformation and grouping, and visualized the data. For further analytical tasks, reference each field, column, and record set explicitly by their Croissant `@id`._